# 2 What did that capacity buy you?

At the end of this notebook you can measure how the score moves with parameters, with
data and with compute on a frozen representation, draw each as a curve with its spread
between seeds, and say on which axis a problem is **bound**: which resource, given more, still raises
the score. That decides where your next
hour goes: a bigger head, more labelled rows, longer training or another encoder. You
can also read a scaling-law figure: a straight line on log-log axes whose slope says
what ten times more buys, valid only inside the range that was measured. You will see
that the slope differs per domain, measure it on two problems of your own, and read
from a measured ratio how much more data a bigger model needs.

Notebook 01 put a gate, a highway and a mixture of experts next to plain models with the
same number of parameters, and no mechanism beat its equal-weight control. This notebook asks the general version of that question. You can spend
resources in three places: more **parameters**, more **data**, more **compute**. Each
costs something, and the folklore says each buys accuracy. So:

> Turn each knob over one to two orders of magnitude and **draw the curve**. Where does
> the metric move, and where are you paying for nothing?

A **scaling curve** is that measurement: the score against one resource, with the others
held fixed, each point trained over several seeds so you can see what exceeds the spread.
The scaling-laws papers (2.7 and 2.8) draw them at millions of dollars per point. On a cached
representation a point costs a second, so you draw them yourself, and the answers on
*your* data are not the answers in the papers.

**Three splits.** Every choice in this notebook (the epoch count of 2.1, each head's epoch
and the head to prefer in 2.2, each encoder's in 2.5) is made on a **valid** split cut from the cache's train split, as in
`1_representation_and_head/03_training`. Every curve from 2.2 on and every number quoted
about it is scored on the cache's **test** split, which no choice has looked at. The first
cell loads DTD (textures, 47 classes, the dataset of `1_representation_and_head/04_search`)
and shows the three splits.

In [ ]:
import sys
import time

import datasets
import matplotlib.pyplot as plt
import numpy as np
import torch
from loguru import logger
from notebooktester import param
from torch import nn
from vectormesh import VectorCache
from vectormesh.components import NeuralNet

logger.remove()
logger.add(sys.stderr, level="WARNING")
datasets.disable_progress_bars()

SEEDS = param((0, 1, 2, 3, 4), test=(0,))
MAX_EPOCHS = param(80, test=2)  # the longest run of 2.1; EPOCHS is chosen inside it
BATCH, LR = 128, 1e-3


def balanced_order(y: torch.Tensor, seed: int = 0) -> torch.Tensor:
    """An order of the rows in which every prefix holds each class about equally often.

    The rows are ranked within their class at random and sorted on that rank, so rows[:n]
    is a stratified subset for every n, and a smaller subset is part of every larger one.
    """
    g = torch.Generator().manual_seed(seed)
    rank = torch.empty(len(y))
    for c in y.unique():
        idx = (y == c).nonzero().squeeze(1)
        rank[idx[torch.randperm(len(idx), generator=g)]] = torch.arange(
            len(idx), dtype=torch.float
        )
    return torch.argsort(rank + 0.5 * torch.rand(len(y), generator=g))


def load(dataset: str, encoder: str) -> dict[str, tuple[torch.Tensor, torch.Tensor]]:
    """The cache's train split cut 80/20 into train and valid, and its own test split."""
    hub = f"pttrn-io/{dataset}-{encoder}"
    train = VectorCache.from_hub(hub, split="train").dataset
    test = VectorCache.from_hub(hub, split="test").dataset
    split = train.train_test_split(test_size=0.2, seed=42, stratify_by_column="label")
    out = {}
    for name, part in [
        ("train", split["train"]),
        ("valid", split["test"]),
        ("test", test),
    ]:
        part = part.with_format("numpy")
        X = torch.tensor(np.asarray(part["embed"], dtype="float32"))
        y = torch.tensor(np.asarray(part["label"]))
        out[name] = (X, y)
    order = balanced_order(out["train"][1])
    out["train"] = (out["train"][0][order], out["train"][1][order])
    return out


dtd = load("dtd", "dinov2-small")
DIM = dtd["train"][0].shape[1]
K = int(dtd["train"][1].max()) + 1

print(f"DTD, dinov2-small: {DIM}-dim vectors, {K} classes")
for name, (X, y) in dtd.items():
    per_class = torch.bincount(y, minlength=K)
    print(
        f"  {name:5s} {tuple(X.shape)}   {int(per_class.min())} to "
        f"{int(per_class.max())} rows per class"
    )
first = dtd["train"][1][: 2 * K]
print(f"the first {2 * K} train rows hold {len(first.unique())} of the {K} classes")

The train rows are put in a **balanced order**: every prefix holds each class about
equally often, so the first 150 rows are a stratified subset of 3 or 4 rows per class,
and a smaller subset is always part of a larger one. 2.3 trains on prefixes; without the
order, a prefix of a cache stored class by class would hold only its first classes, and
the curve would measure how many classes you had.

The next cell holds the tools every section uses. `train` trains a head with Adam, batch
128, for a number of **epochs** (passes over the train rows) or a number of **optimizer
steps** (one step is one batch: a forward pass, a backward pass, one update of the
weights), and after every epoch records the steps and training seconds so far, the train
and valid loss and the valid and test accuracy. `repeat` runs it once per seed.
`choose_epochs` is the rule that picks an epoch count from the valid curve (2.1).

**Is a difference a result?** The rule of lesson 2: divide the difference of two means by
its **standard error**, $\sqrt{s_a^2/n_a + s_b^2/n_b}$ with $s$ the std over the seeds. Within
2 standard errors it is no result, from 2 to 3 marginal, beyond 3 a result. When both means
come from the same runs (one head read at two epochs), the difference is taken per seed and
its standard error is the std of those differences over $\sqrt{n}$ (paired). `say` prints
one comparison in that form. The ± after a mean is the spread between seeds (`np.std`, which divides by
$n$); the standard error uses the sample std, which divides by $n - 1$ and is about 12%
larger with five seeds. The ± is not the yardstick. The seeds vary the initial weights and the order of the batches, not the
train/valid split or which rows a subset holds, so that spread is a lower bound on how far
a point could move.

In [ ]:
def accuracy(model: nn.Module, X: torch.Tensor, y: torch.Tensor) -> float:
    model.eval()
    with torch.no_grad():
        return float((model(X).argmax(1) == y).float().mean())


def n_params(model: nn.Module) -> int:
    return sum(p.numel() for p in model.parameters())


def train(
    make,
    d,
    epochs: int | None = None,
    steps: int | None = None,
    rows: int | None = None,
):
    """Train make() on the first `rows` train rows, for `epochs` epochs or `steps` optimizer
    steps. After every epoch (and at the last step) it records the optimizer steps and the
    training seconds so far, the train and valid loss, and the valid and test accuracy."""
    X, y = d["train"]
    if rows is not None:
        X, y = X[:rows], y[:rows]
    model = make()
    opt = torch.optim.Adam(model.parameters(), lr=LR)
    lossf = nn.CrossEntropyLoss()
    h = {
        k: [] for k in ("steps", "seconds", "train_loss", "valid_loss", "valid", "test")
    }
    done, seconds, epoch = 0, 0.0, 0
    while (epochs is None or epoch < epochs) and (steps is None or done < steps):
        model.train()
        start, total, seen = time.perf_counter(), 0.0, 0
        perm = torch.randperm(len(X))
        for i in range(0, len(X), BATCH):
            idx = perm[i : i + BATCH]
            opt.zero_grad()
            loss = lossf(model(X[idx]), y[idx])
            loss.backward()
            opt.step()
            done += 1
            total, seen = total + loss.item() * len(idx), seen + len(idx)
            if steps is not None and done == steps:
                break
        seconds += time.perf_counter() - start
        epoch += 1
        Xv, yv = d["valid"]
        model.eval()
        with torch.no_grad():
            valid_loss = float(lossf(model(Xv), yv))
        for k, v in [
            ("steps", done),
            ("seconds", seconds),
            ("train_loss", total / seen),
            ("valid_loss", valid_loss),
            ("valid", accuracy(model, Xv, yv)),
            ("test", accuracy(model, *d["test"])),
        ]:
            h[k].append(v)
    return model, h


def repeat(make, d, **budget) -> dict[str, np.ndarray]:
    """train() once per seed; every record becomes an array of (seeds, epochs)."""
    runs = []
    for s in SEEDS:
        torch.manual_seed(s)
        runs.append(train(make, d, **budget)[1])
    out = {k: np.array([r[k] for r in runs]) for k in runs[0]}
    out["params"] = n_params(make())
    return out


def choose_epochs(valid: np.ndarray) -> int:
    """The epoch with the highest valid accuracy, averaged over the seeds; numbered from 1."""
    return int(valid.mean(0).argmax()) + 1


def at_epoch(r: dict, epoch: int, key: str = "test") -> tuple[float, float]:
    """Mean and std over the seeds at one epoch."""
    return float(r[key][:, epoch - 1].mean()), float(r[key][:, epoch - 1].std())


def final(r: dict, key: str = "test") -> tuple[float, float]:
    """Mean and std over the seeds of the last record."""
    return float(r[key][:, -1].mean()), float(r[key][:, -1].std())


def compare(a, b, paired: bool = False) -> tuple[float, float, float]:
    """Difference of the means b - a over the seeds, its standard error and their ratio.

    Unpaired: sqrt(var_a / n_a + var_b / n_b), as in lesson 2. Paired (the same runs read
    twice): the std of the per-seed differences over sqrt(n). Sample variances (ddof=1); with
    one seed there is no standard error, and the ratio is nan.
    """
    a, b = np.asarray(a, dtype=float), np.asarray(b, dtype=float)
    diff = float(b.mean() - a.mean())
    if min(len(a), len(b)) < 2:
        return diff, float("nan"), float("nan")
    if paired:
        se = float((b - a).std(ddof=1) / np.sqrt(len(a)))
    else:
        se = float(np.sqrt(a.var(ddof=1) / len(a) + b.var(ddof=1) / len(b)))
    return diff, se, diff / se if se > 0 else float("nan")


def verdict(z: float) -> str:
    """Lesson 2's reading of a difference in standard errors."""
    if np.isnan(z):
        return "no standard error"
    return "no result" if abs(z) < 2 else "marginal" if abs(z) < 3 else "result"


def say(label: str, a, b, paired: bool = False) -> tuple[float, float, float]:
    """Print b - a in points of accuracy, its standard error, the ratio and the verdict."""
    d, se, z = compare(a, b, paired)
    print(
        f"  {label:34s} {100 * d:+6.2f} pt  se {100 * se:4.2f}  {z:+5.1f} se  {verdict(z)}"
    )
    return d, se, z

## 2.1 First, how long is long enough?

**Question.** Every point below is a trained head, and a head trained too briefly
measures its budget rather than the knob the curve claims to be about. So how long is long
enough is settled first, and on valid.

**Setup.** The baseline head, `NeuralNet(384, 47)` (one hidden layer of 384 units),
trained for `MAX_EPOCHS` epochs with five seeds, valid loss and valid accuracy recorded
after every epoch. The rule: `EPOCHS` is the epoch with the highest valid accuracy,
averaged over the seeds. That is the epoch early stopping on valid accuracy keeps
(`1_representation_and_head/03_training`), applied to the mean of five runs.

**Expect.** Write down before you run it: at what epoch does valid accuracy stop rising,
and does valid loss do the same?

In [ ]:
base = repeat(lambda: NeuralNet(DIM, K), dtd, epochs=MAX_EPOCHS)
EPOCHS = choose_epochs(base["valid"])

va, va_sd = base["valid"].mean(0), base["valid"].std(0)
print(f"{'epoch':>5s} {'train loss':>10s} {'valid loss':>10s} {'valid acc':>18s}")
for e in [1, 2, 5, 10, 20, 40, 80]:
    if e <= MAX_EPOCHS:
        print(
            f"{e:5d} {base['train_loss'][:, e - 1].mean():10.4f} "
            f"{base['valid_loss'][:, e - 1].mean():10.4f} "
            f"{va[e - 1]:10.4f} +- {va_sd[e - 1]:.4f}"
        )
print(
    f"EPOCHS = {EPOCHS}: the highest mean valid accuracy, "
    f"{va[EPOCHS - 1]:.4f} +- {va_sd[EPOCHS - 1]:.4f}"
)
low = int(base["valid_loss"].mean(0).argmin()) + 1
print(f"lowest mean valid loss at epoch {low}")
E10 = min(10, MAX_EPOCHS)
print("valid accuracy, the same runs at two epochs (paired):")
say(
    f"epoch {E10} -> {EPOCHS}",
    base["valid"][:, E10 - 1],
    base["valid"][:, EPOCHS - 1],
    paired=True,
)
say(
    f"epoch {EPOCHS} -> {MAX_EPOCHS}",
    base["valid"][:, EPOCHS - 1],
    base["valid"][:, -1],
    paired=True,
)

ep = np.arange(1, MAX_EPOCHS + 1)
fig, (a1, a2) = plt.subplots(1, 2, figsize=(12, 3.8))
a1.plot(ep, base["train_loss"].mean(0), label="train loss")
a1.plot(ep, base["valid_loss"].mean(0), label="valid loss")
a1.set_xscale("log")
a1.set_xlabel("epoch (log)")
a1.set_ylabel("cross-entropy loss")
a1.set_title("Train loss keeps falling; valid loss turns up")
a2.plot(ep, va, color="tab:orange", label="valid accuracy")
a2.fill_between(
    ep,
    va - va_sd,
    va + va_sd,
    color="tab:orange",
    alpha=0.25,
    label="+- 1 std over seeds",
)
a2.set_xscale("log")
a2.set_xlabel("epoch (log)")
a2.set_ylabel("valid accuracy")
a2.set_title("Valid accuracy: where the plateau starts")
for a in (a1, a2):
    a.axvline(EPOCHS, color="grey", ls="--", label=f"EPOCHS = {EPOCHS}")
    a.grid(alpha=0.3)
    a.legend(frameon=False)
plt.tight_layout()
plt.show()

**Reading.** Valid accuracy rises steeply to epoch 5 (0.7872), creeps to its highest at
epoch 20 (0.8000 ± 0.0063) and then stays there: 0.7955 at epoch 80. From epoch 10 to 20 the mean valid accuracy rises 0.27 points, no result
(1.4 se, paired); from 20 to 80 it falls 0.45 points, marginal (−2.0 se). Any epoch from
10 on is about as good; 20 is the rule's pick, not a sharp optimum.

Valid **loss** tells another story: lowest at epoch 5, then rising to 1.01 at epoch 80,
while train loss falls to 0.0005. The head memorises the train rows and grows more
confident on the valid rows it gets wrong, but which rows it gets right hardly changes.
Loss and accuracy pick different epochs; the number this notebook reports is accuracy,
so the epoch is chosen on accuracy.

**`EPOCHS = 20`** for the data axis below, chosen on valid. These runs also recorded
their test accuracy; it is read in 2.4 and played no part in this choice.

> `5_unsupervised/01_why_pretext_tasks_work` §1.3 runs a check like this on an encoder it
> trains itself and finds the opposite: there the accuracy on unseen classes keeps moving
> with the number of epochs, and a short run hides the effect. Which is why the rule is
> "check", not "train longer".

## 2.2 Axis one: parameters

**Question.** Does a bigger head score higher? "Bigger" is two different things, so they
are measured apart: **width** (one hidden layer, from none to 1024 units) and **depth**
(width 256, one to four hidden layers). A model that is wider and deeper at once could not
say which of the two moved it.

**Setup.** Data fixed: the 3008 train rows, five seeds per model, scored on test, each
head at its own epoch chosen on valid (see the check after the result). Every row prints
its parameter count and that epoch.

**Expect.** Write down: is the 1024-wide head better than the linear one, by more than the
spread? And the four-layer head?

In [ ]:
def head(*hidden: int):
    """NeuralNet with these hidden widths; no widths is one Linear layer."""
    if not hidden:
        return lambda: nn.Linear(DIM, K)
    return lambda: NeuralNet([DIM, *hidden], K)


WIDTH = [("linear", head())] + [
    (f"1 x {w}", head(w)) for w in (64, 128, 256, 512, 1024)
]
DEPTH = [(f"{n} x 256", head(*[256] * n)) for n in (1, 2, 3, 4)]

param_axis = {}  # name -> (params, its own epoch, test mean, test std, the runs)
for series, models in [("width", WIDTH), ("depth", DEPTH)]:
    print(f"{series}: {'model':9s} {'params':>10s} {'epochs':>6s}   test accuracy")
    for name, make in models:
        if name not in param_axis:
            r = repeat(make, dtd, epochs=MAX_EPOCHS)
            e = choose_epochs(r["valid"])
            param_axis[name] = (r["params"], e, *at_epoch(r, e), r)
        p, e, m, s, _ = param_axis[name]
        print(f"       {name:9s} {p:10,d} {e:6d}   {m:.4f} +- {s:.4f}")


def runs_at_own_epoch(name: str, key: str = "test") -> np.ndarray:
    """One head's accuracy per seed, at its own epoch chosen on valid."""
    r, e = param_axis[name][4], param_axis[name][1]
    return r[key][:, e - 1]


print("test accuracy, one head against another:")
for a, b in [
    ("linear", "1 x 1024"),
    ("linear", "1 x 64"),
    ("1 x 512", "1 x 1024"),
    ("1 x 256", "2 x 256"),
    ("2 x 256", "3 x 256"),
    ("3 x 256", "4 x 256"),
    ("linear", "4 x 256"),
    ("1 x 128", "4 x 256"),
]:
    say(f"{a} -> {b}", runs_at_own_epoch(a), runs_at_own_epoch(b))

fig, (a1, a2) = plt.subplots(1, 2, figsize=(12, 3.8), sharey=True)
for a, models, x, xlabel in [
    (a1, WIDTH, [param_axis[n][0] for n, _ in WIDTH], "parameters (log)"),
    (a2, DEPTH, [1, 2, 3, 4], "hidden layers of width 256"),
]:
    ms = [param_axis[n][2] for n, _ in models]
    ss = [param_axis[n][3] for n, _ in models]
    a.errorbar(x, ms, yerr=ss, fmt="o-", capsize=3)
    for (n, _), xi, mi in zip(models, x, ms):
        a.annotate(
            f"{n}\n{param_axis[n][0]:,}",
            (xi, mi),
            fontsize=7,
            xytext=(4, -18),
            textcoords="offset points",
        )
    a.set_xlabel(xlabel)
    a.grid(alpha=0.3)
a1.set_xscale("log")
a1.set_ylabel("test accuracy (at each head's own epoch)")
a2.set_xticks([1, 2, 3, 4])
a1.set_title("Width: one hidden layer, wider")
a2.set_title("Depth: width 256, more layers")
plt.tight_layout()
plt.show()

**Reading.** Width buys a little. From the bare linear map (0.7768 ± 0.0050,
18,095 parameters) to `1 x 1024` (0.8023 ± 0.0019, 442,415) is +2.55 points, 9.5 standard
errors: a result. Doubling from `1 x 512` (0.8007) to `1 x 1024` adds 0.16 points, no result
(0.9 se), and `1 x 64` (0.7780) is no result against the linear map (0.3 se).

Depth hands it back. `2 x 256` (0.7920) is 0.43 points below `1 x 256` (0.7963), marginal
(−2.2 se); the third layer costs 1.04 points (−6.1 se, a result) and the fourth 1.02 more
(−2.6 se, marginal; `4 x 256` has the largest spread, ± 0.0075). `4 x 256` ends no result
against the linear map (−1.2 se) at seventeen times its parameters. The parameter count
alone does not order the models: `4 x 256` (308,015) scores 1.97 points below `1 x 128`
(55,343), a result (−5.1 se).

So on this axis there are about two and a half points, collected by width up to `1 x 512`,
and more parameters stacked as depth lose them again. Keep that number in mind: the next
axis moves by twenty.

Read it with what the head sits on, and with the recipe: one learning rate for every head,
and no regularisation. The encoder already did the work; the head reads off an answer that
is already in the 384 numbers. Depth cost points here; whether that is overfitting, this
notebook does not test. The output even argues against the simplest version: the widest
head, with more weights than any deep one, scores best, and the valid loss turns up after a
few epochs for every head drawn below, not only for the deep one.

**Is the comparison fair?** A wider or a deeper head could need longer than the baseline
head, and a row that stops it early measures a head that had not finished. So no head above
got `EPOCHS`: each was trained for `MAX_EPOCHS`, its own epoch was chosen on valid by the
rule of 2.1 (the `epochs` column, from 9 to 35), and its test accuracy is read at that
epoch. The curves of the smallest, the widest and the deepest head show what that choice
looks like.

In [ ]:
fig, (a1, a2) = plt.subplots(1, 2, figsize=(12, 3.8))
for i, n in enumerate(["linear", "1 x 1024", "4 x 256"]):
    r, e = param_axis[n][4], param_axis[n][1]
    a1.plot(ep, r["valid_loss"].mean(0), color=f"C{i}", label=n)
    a2.plot(ep, r["valid"].mean(0), color=f"C{i}", label=f"{n} (epoch {e})")
    a2.plot(e, r["valid"].mean(0)[e - 1], "o", color=f"C{i}")
a1.set_ylabel("valid loss")
a1.set_title("Valid loss of the smallest, widest and deepest head")
a2.set_ylabel("valid accuracy")
a2.set_title("Valid accuracy; the dot is each head's own epoch")
for a in (a1, a2):
    a.set_xscale("log")
    a.set_xlabel("epoch (log)")
    a.grid(alpha=0.3)
    a.legend(frameon=False, fontsize=8)
plt.tight_layout()
plt.show()

Every valid-accuracy curve has flattened by its dot and stays flat to `MAX_EPOCHS`: no
head was scored while it was still learning. The valid loss turns up after a few epochs for
all three, steepest for `4 x 256`, which climbs to 1.7, and its accuracy sits lowest the
whole way.

**Beyond the top score.** The highest mean on this axis is not the whole answer. A head
that is no result against the best, with fewer parameters, does the same job for less. The
rule in the next cell is a choice, so it is made on **valid**: take the head with the
highest mean valid accuracy (each at its own epoch), keep every head whose difference with
it is within 2 standard errors, and pick the one with the fewest parameters. Only then are
the best and the pick read on test, once. The cell also times both on this machine: a
forward pass over the 1880 test rows, repeated, the median and the middle half of the
repeats.

**Expect.** Write down which head the rule picks, and whether half the parameters makes it
twice as fast.

In [ ]:
REPEATS = param(50, test=3)


def latency_ms(make, X: torch.Tensor) -> tuple[float, float, float]:
    # Median, 25th and 75th percentile of the time of one forward pass over X, in ms.
    torch.manual_seed(0)
    model = make().eval()
    times = []
    with torch.no_grad():
        model(X)  # the first call allocates; it is not timed
        for _ in range(REPEATS):
            start = time.perf_counter()
            model(X)
            times.append(1000 * (time.perf_counter() - start))
    return tuple(float(v) for v in np.percentile(times, [50, 25, 75]))


makers = dict(WIDTH + DEPTH)
best = max(param_axis, key=lambda n: runs_at_own_epoch(n, "valid").mean())
print(f"best on valid: {best}; every head against it, valid accuracy:")
kept = [best]
for n in param_axis:
    if n != best:
        _, _, z = say(
            f"{best} -> {n}",
            runs_at_own_epoch(best, "valid"),
            runs_at_own_epoch(n, "valid"),
        )
        if abs(z) < 2:
            kept.append(n)
pick = min(kept, key=lambda n: param_axis[n][0])
print(
    f"no result against {best} on valid: {', '.join(kept)};  the fewest parameters: {pick}\n"
)

print("test accuracy, read once:")
say(f"{best} -> {pick}", runs_at_own_epoch(best), runs_at_own_epoch(pick))
print(
    f"{'head':9s} {'params':>9s} {'test accuracy':>17s} {'ms per pass (25-75%)':>24s}"
)
medians = {}
for n in dict.fromkeys([best, pick]):
    med, lo, hi = latency_ms(makers[n], dtd["test"][0])
    medians[n] = med
    p, _, m, s, _ = param_axis[n]
    print(f"{n:9s} {p:9,d} {m:9.4f} +- {s:.4f} {med:9.2f} ({lo:.2f} to {hi:.2f})")
print(
    f"{best} takes {medians[best] / medians[pick]:.1f} times as long per pass as {pick}"
)

**Reading.** On valid, `1 x 1024` is best, and two heads are no result
against it: `1 x 512` (−0.35 points, −1.1 se) and `1 x 256` (−0.48 points, −1.3 se). The
rule picks the smaller, `1 x 256`, with a quarter of the parameters (110,639 against
442,415).

Read on test, once: `1 x 256` scores 0.7963 ± 0.0027 against 0.8023 ± 0.0019, 0.61 points
lower, a result (−3.7 se). One possible reason, not tested here: the valid split holds
752 rows and the test split 1880, so valid separates small gaps less well. That is the cost of the choice,
and it is reported, not undone: choosing again on test would make the test number a choice
too.

On the machine that ran this notebook, one pass over the 1880 test rows took `1 x 1024`
about three times as long as `1 x 256` (the ratio is printed under
the table), and the middle halves of the repeats do not overlap.
The milliseconds change from run to run, more so when the machine is busy; read them in the
table.

So the conclusion goes beyond the top score, with its price stated: **`1 x 256` answers
about three times as fast for 0.6 points of test accuracy.** Where speed or memory counts,
that can be the trade to make. Two limits. The timing holds for this machine and a batch of
1880 rows; on other hardware, or one row at a time, it is measured again. And nothing here
measured energy: fewer operations suggest less, but a claim about energy needs an energy
measurement.

## 2.3 Axis two: data

**Question.** Does the score move with the number of labelled rows?

**Setup.** The baseline head on 5%, 10%, 25%, 50% and 100% of the train rows (balanced
prefixes), five seeds each, scored on test. One subtlety decides what the curve means.
At a fixed number of epochs, 5% of the rows is 2 batches per epoch and 100% is 24, so the
small subset also gets about a twelfth of the optimizer steps: data and compute shrink
together. So the axis is measured twice:

- **equal epochs**: every subset gets `EPOCHS` passes. This is what you get when you keep
  your recipe and collect fewer rows: data and compute change together.
- **equal steps**: every subset gets the optimizer steps of the full set at `EPOCHS`
  epochs, so a small subset is passed over many more times. This holds compute fixed and
  changes only the data.

The steps and training seconds of every point are printed beside it.

**Expect.** Write down: how far apart are 5% and 100%? And does giving the small subsets
the full number of steps close the gap?

In [ ]:
FRACTIONS = param([0.05, 0.1, 0.25, 0.5, 1.0], test=[0.25, 1.0])


def data_axis(d, epochs: int, label: str) -> list[tuple]:
    """Test accuracy over fractions of the train rows, at equal epochs and at equal steps."""
    n_full = len(d["train"][0])
    k = int(d["train"][1].max()) + 1
    dim = d["train"][0].shape[1]
    steps_full = epochs * -(-n_full // BATCH)  # ceil: a last short batch is a step too
    rows = []
    print(
        f"{label}: {'rows':>5s} {'per class':>9s} | {'steps':>5s} {'seconds':>7s} "
        f"{'equal epochs':>16s} | {'steps':>5s} {'seconds':>7s} {'equal steps':>16s}"
    )
    for frac in FRACTIONS:
        n = int(frac * n_full)
        same_epochs = repeat(lambda: NeuralNet(dim, k), d, epochs=epochs, rows=n)
        same_steps = repeat(lambda: NeuralNet(dim, k), d, steps=steps_full, rows=n)
        row = (
            n,
            *final(same_epochs),
            *final(same_steps),
            int(same_epochs["steps"][0, -1]),
            int(same_steps["steps"][0, -1]),
            float(same_epochs["seconds"][:, -1].mean()),
            float(same_steps["seconds"][:, -1].mean()),
            same_epochs["test"][:, -1],
            same_steps["test"][:, -1],
        )
        rows.append(row)
        print(
            f"{'':{len(label) + 1}s} {n:5d} {n / k:9.1f} | {row[5]:5d} {row[7]:7.2f} "
            f"{row[1]:8.4f} +- {row[2]:.4f} | {row[6]:5d} {row[8]:7.2f} {row[3]:8.4f} +- {row[4]:.4f}"
        )
    print("test accuracy, equal steps against equal epochs at each size:")
    for r in rows:
        say(f"{r[0]} rows", r[9], r[10])
    print("the last doubling of the rows:")
    say("equal epochs", rows[-2][9], rows[-1][9])
    say("equal steps", rows[-2][10], rows[-1][10])
    return rows


def plot_data_axis(rows, title):
    n = [r[0] for r in rows]
    plt.figure(figsize=(7, 3.8))
    plt.errorbar(
        n,
        [r[1] for r in rows],
        yerr=[r[2] for r in rows],
        fmt="o-",
        capsize=3,
        label="equal epochs: fewer rows, fewer steps",
    )
    plt.errorbar(
        n,
        [r[3] for r in rows],
        yerr=[r[4] for r in rows],
        fmt="s--",
        capsize=3,
        label="equal steps: fewer rows, more passes over them",
    )
    plt.xscale("log")
    plt.xlabel("training rows (log)")
    plt.ylabel("test accuracy")
    plt.title(title)
    plt.grid(alpha=0.3)
    plt.legend(frameon=False)
    plt.tight_layout()
    plt.show()


dtd_data = data_axis(dtd, EPOCHS, "DTD")
plot_data_axis(dtd_data, "DTD: the data axis at equal epochs and at equal steps")

**Reading.** At equal epochs the curve runs from 0.5798 ± 0.0093 at 150 rows
(3.2 per class, 40 steps) to 0.7997 ± 0.0042 at 3008 rows (480 steps): 22 points, eight
times the whole parameter axis.

Is that data, or the twelve-fold difference in steps? The equal-steps arm answers it.
Giving the 150 rows all 480 steps (about 240 passes over them) lifts them by 1.47 points,
marginal (2.3 se); from 300 rows on, the extra steps are no result. At equal compute the
curve still spans 20.5 points. Almost all of the movement is data.

Each doubling is worth a little less than the one before: on the equal-steps arm 5% to 10%
is +7.4 points, 50% to 100% is +3.1, still a result (10.9 se). **On DTD, more labelled rows
has not stopped paying at the full train split.** 2.9 draws this curve on the axes the
scaling papers use.

Two consequences:

- **Where you are on this curve decides what advice is right.** At 150 rows the answer to
  "how do I improve this model" is *get more data*, and no head will save you.
- **The steep part is where a frozen encoder earns its keep.** At 3 rows per class the head
  is near 0.58, against a chance level of 1/47 ≈ 0.02, and that is the encoder's doing.

## 2.4 Axis three: compute

**Question.** Does training longer pay?

**Setup.** Compute here is **epochs of one head**: the baseline head of 2.1, on the full
train rows, measured as optimizer steps and training seconds (the time in the training
steps, evaluation not counted). It is not a bigger encoder or a bigger batch; those are
other ways to spend compute. The runs are the ones of 2.1: their valid curve chose
`EPOCHS`, and this section reads their **test** accuracy, which played no part in that
choice.

**Expect.** Write down: how many optimizer steps until the test accuracy stops rising,
and what happens between `EPOCHS` and `MAX_EPOCHS`?

In [ ]:
te, te_sd = base["test"].mean(0), base["test"].std(0)
steps_per_epoch = int(base["steps"][0, 0])
print(f"the head of 2.1, {steps_per_epoch} optimizer steps per epoch")
print(f"{'epoch':>5s} {'steps':>6s} {'seconds':>7s} {'test acc':>18s}")
for e in [1, 2, 5, 10, 20, 40, 80]:
    if e <= MAX_EPOCHS:
        print(
            f"{e:5d} {int(base['steps'][0, e - 1]):6d} {base['seconds'][:, e - 1].mean():7.2f} "
            f"{te[e - 1]:10.4f} +- {te_sd[e - 1]:.4f}"
        )
after = te[EPOCHS - 1 :]
print(
    f"from epoch {EPOCHS} to {MAX_EPOCHS}: test accuracy between {after.min():.4f} and {after.max():.4f}"
)
print("test accuracy, the same runs at two epochs (paired):")
say(
    f"epoch {E10} -> {MAX_EPOCHS}",
    base["test"][:, E10 - 1],
    base["test"][:, -1],
    paired=True,
)
say(
    f"epoch {EPOCHS} -> {MAX_EPOCHS}",
    base["test"][:, EPOCHS - 1],
    base["test"][:, -1],
    paired=True,
)

fig, (a1, a2) = plt.subplots(1, 2, figsize=(12, 3.8), sharey=True)
for a, x, xlabel in [
    (a1, base["steps"][0], "optimizer steps (log)"),
    (a2, base["seconds"].mean(0), "training seconds (log)"),
]:
    a.plot(x, te, color="tab:green")
    a.fill_between(x, te - te_sd, te + te_sd, color="tab:green", alpha=0.25)
    a.axvline(x[EPOCHS - 1], color="grey", ls="--", label=f"epoch {EPOCHS}")
    a.set_xscale("log")
    a.set_xlabel(xlabel)
    a.grid(alpha=0.3)
    a.legend(frameon=False)
a1.set_ylabel("test accuracy")
a1.set_title("Compute in optimizer steps")
a2.set_title("The same runs in seconds")
plt.tight_layout()
plt.show()

**Reading.** Compute pays at first and then stops. One epoch (24 steps) gives
0.7367; epoch 5 (120 steps) 0.7866; epoch 20 (480 steps) 0.7997. From epoch 10 to 80 the
head still gains 0.57 points, a result (4.5 se, paired); from epoch 20 to 80, four times the
steps, it gains 0.11, no result (1.4 se). So compute is flat from epoch 20, and the test
curve agrees with the valid curve of 2.1, which chose epoch 20 without seeing it.

On a cached representation the whole axis costs a few seconds of training, which is why it
is cheap to measure, and here it is not the axis that binds: past epoch 20 the head has
used what the vectors hold.

Read the seconds for what they are: **training time of this head on the machine that ran
this notebook**; they change a little from run to run, so read them in the table. They do
not say how fast a trained head answers: a claim about latency needs inference timed on the
same hardware the claim is about. And they do not say what the training cost in energy: a
claim about energy needs an energy measurement. Kaplan's compute in 2.7 is counted in
floating point operations (additions and multiplications) instead, which does not depend on
the machine, and says nothing about seconds or energy either.

## 2.5 The axis that is not capacity at all: the encoder

**Question.** Parameters and compute buy little on top of a frozen representation. What
does another representation buy?

**Setup.** The baseline head on three caches of the same DTD images: `dinov2-small`
(384-dim), `resnet-18` (512) and `mobilenet_v2` (1280). Each encoder gets its own train,
valid and test split of its own cache, its own epoch count chosen on valid by the rule of
2.1, five seeds, scored on test. Its head's parameter count grows with its dimension.

**Expect.** Write down: does the encoder with the largest head win, and is the encoder
axis wider or narrower than the parameter axis of 2.2?

In [ ]:
ENCODERS = param(
    ["dinov2-small", "resnet-18", "mobilenet_v2_1.0_224"], test=["dinov2-small"]
)


def encoder_axis(dataset: str) -> list[tuple]:
    """Per encoder: the baseline head, its own epoch chosen on valid, scored on test."""
    rows = []
    print(
        f"{dataset}: {'encoder':22s} {'dim':>5s} {'head params':>11s} {'epochs':>6s}   test accuracy"
    )
    for enc in ENCODERS:
        d = load(dataset, enc)
        dim, k = d["train"][0].shape[1], int(d["train"][1].max()) + 1
        r = repeat(lambda dim=dim, k=k: NeuralNet(dim, k), d, epochs=MAX_EPOCHS)
        e = choose_epochs(r["valid"])
        m, s = at_epoch(r, e)
        rows.append((enc, dim, r["params"], e, m, s, r["test"][:, e - 1]))
        print(
            f"{'':{len(dataset) + 1}s} {enc:22s} {dim:5d} {r['params']:11,d} {e:6d}   {m:.4f} +- {s:.4f}"
        )
    return rows


dtd_enc = encoder_axis("dtd")

**Reading.** `dinov2-small` scores 0.7997 ± 0.0042, `resnet-18` 0.6482 ± 0.0033 (at its
own epoch 41) and `mobilenet_v2` 0.6589 ± 0.0021: **15.1 points**, six times the whole
parameter axis of 2.2, from a choice that adds no labelled rows and no epochs (each
encoder ran once, when its cache was built).

And look at the parameter column. `mobilenet_v2`'s 1280 dimensions give its head 1,699,887
parameters, ten times the 165,935 of `dinov2-small`'s, and it loses by 14 points. If
parameters were the resource that mattered, that row would not exist.

One fact colours this axis. DINOv2's pretraining set, LVD-142M, holds a million web images
retrieved because they resemble DTD's training images (Oquab et al. 2023, Table 15); EuroSAT
is not among its sources. Near-duplicates of benchmark test images were removed (their
Section 3), so this is no test leakage, but `dinov2-small` was curated towards textures, and
part of its lead on DTD, and of the contrast with EuroSAT in 2.6, can come from that.

## 2.6 Another problem, another binding axis

**Question.** Goal of this notebook: the axis that pays differs per problem. Is that so,
or is it a DTD result?

**Setup.** EuroSAT (satellite tiles, 10 classes, 6000 rows in its train split, the dataset of
`1_representation_and_head/03_training`) with the same protocol: an epoch count chosen on
its own valid split by the rule of 2.1, the data axis at equal epochs and equal steps, the
encoder axis over the same three encoders, five seeds, scored on test. In lesson 1 its
encoders landed within a few points of each other near 0.95; on DTD they spread over 15
points (2.5). EuroSAT has less headroom.

**Expect.** Write down: which axis binds on EuroSAT?

In [ ]:
eurosat = load("eurosat", "dinov2-small")
E_DIM = eurosat["train"][0].shape[1]
E_K = int(eurosat["train"][1].max()) + 1
for name, (X, y) in eurosat.items():
    print(f"EuroSAT {name:5s} {tuple(X.shape)}")

e_base = repeat(lambda: NeuralNet(E_DIM, E_K), eurosat, epochs=MAX_EPOCHS)
E_EPOCHS = choose_epochs(e_base["valid"])
print(f"EuroSAT: E_EPOCHS = {E_EPOCHS}, chosen on valid by the rule of 2.1")
print(f"{'epoch':>5s} {'steps':>6s} {'valid acc':>18s} {'test acc':>18s}")
for e in [1, 2, 5, 10, 20, 40, 80]:
    if e <= MAX_EPOCHS:
        v, vs = at_epoch(e_base, e, "valid")
        t, ts = at_epoch(e_base, e)
        print(
            f"{e:5d} {int(e_base['steps'][0, e - 1]):6d} {v:10.4f} +- {vs:.4f} {t:10.4f} +- {ts:.4f}"
        )

print("valid accuracy, the same runs at two epochs (paired):")
say(
    f"epoch {E10} -> {E_EPOCHS}",
    e_base["valid"][:, E10 - 1],
    e_base["valid"][:, E_EPOCHS - 1],
    paired=True,
)

eurosat_data = data_axis(eurosat, E_EPOCHS, "EuroSAT")
plot_data_axis(
    eurosat_data, "EuroSAT: the data axis at equal epochs and at equal steps"
)
eurosat_enc = encoder_axis("eurosat")

**Reading.** EuroSAT starts high: one epoch already gives 0.9270 on test. Its
valid accuracy moves little after epoch 10: from 10 to the chosen 78 it gains 0.35 points,
no result (1.9 se), so the rule picks epoch 78 on a plateau where every epoch scores about
the same.

The data axis runs from 0.8897 at 240 rows (24 per class) to 0.9484 at 4800, 5.9 points,
and the equal-steps arm again shows it is data, not steps: the extra steps are no result at
every size. The last doubling buys +0.43 points at equal steps, a result (3.5 se), and +0.40
at equal epochs, marginal (2.9 se): data still pays on EuroSAT, but a seventh of what it
pays on DTD (+3.1). The encoder axis spans 3.1 points (`dinov2-small` 0.9484, `resnet-18`
0.9309, `mobilenet_v2` 0.9176), against 15.1 on DTD.

### The axes side by side

The **span** of an axis is its highest mean test accuracy minus its lowest. Whether an axis
still pays is read from its **last step**, judged by the rule: width from 512 to 1024 units,
depth from 3 to 4 layers, data from 50% to 100% of the rows, compute from epoch 20 (the
epoch 2.1 chose) to `MAX_EPOCHS` (the same runs, paired). An axis **binds** where its last
step still pays: it is the resource that holds the score back, what 2.7 calls the
bottleneck. The encoder has no order, so its step is the best encoder against the second
best; it says how much the choice of encoder mattered, not whether more of something still
pays, so it is read apart from "binds". The chart's x axis is symlog: linear between −1 and
1 point and logarithmic beyond, so a 14-point step and a 0.2-point step fit on one axis.

In [ ]:
def axis_row(means, a, b, paired: bool = False) -> tuple:
    """Span of the means, then the step from runs a to runs b: difference, se, ratio; points."""
    d, se, z = compare(a, b, paired)
    return 100 * (max(means) - min(means)), 100 * d, 100 * se, z


def data_rows(rows) -> dict:
    return {
        "data, equal epochs": axis_row([r[1] for r in rows], rows[-2][9], rows[-1][9]),
        "data, equal steps": axis_row([r[3] for r in rows], rows[-2][10], rows[-1][10]),
    }


FROM = min(20, MAX_EPOCHS)  # the epoch 2.1 chose on DTD


def compute_row(r) -> dict:
    m = r["test"].mean(0)[FROM - 1 :]
    step = axis_row(m, r["test"][:, FROM - 1], r["test"][:, -1], paired=True)
    return {f"compute, epoch {FROM} to {MAX_EPOCHS}": step}


def encoder_row(rows) -> dict:
    top = sorted(rows, key=lambda row: row[4])
    top = top if len(top) > 1 else top * 2  # one encoder (test run): a step of 0
    return {
        "apart: encoder, best two": axis_row(
            [row[4] for row in rows], top[-2][6], top[-1][6]
        )
    }


def head_row(models, name: str) -> dict:
    a, b = models[-2][0], models[-1][0]
    means = [param_axis[n][2] for n, _ in models]
    return {name: axis_row(means, runs_at_own_epoch(a), runs_at_own_epoch(b))}


table = {
    "DTD": {
        **head_row(WIDTH, "parameters, width"),
        **head_row(DEPTH, "parameters, depth"),
        **data_rows(dtd_data),
        **compute_row(base),
        **encoder_row(dtd_enc),
    },
    "EuroSAT": {
        **data_rows(eurosat_data),
        **compute_row(e_base),
        **encoder_row(eurosat_enc),
    },
}
print(
    "last step: width 512 -> 1024, depth 3 -> 4 layers, data 50% -> 100%, "
    f"compute epoch {FROM} -> {MAX_EPOCHS}, encoder second best -> best\n"
)
print(
    f"{'':8s} {'axis':24s} {'span':>5s} {'last step':>9s} {'se':>5s} {'ratio':>6s}   verdict"
    "   (points of test accuracy)"
)
for ds, axes in table.items():
    for axis, (sp, d, se, z) in axes.items():
        print(
            f"{ds:8s} {axis:24s} {sp:5.1f} {d:+9.2f} {se:5.2f} {z:+6.1f}   {verdict(z)}"
        )
    print()

fig, axs = plt.subplots(1, 2, figsize=(12, 3.8), sharex=True)
for a, (ds, axes) in zip(axs, table.items()):
    names = list(axes)
    a.barh(
        names,
        [axes[n][1] for n in names],
        xerr=[2 * axes[n][2] for n in names],
        color="tab:blue",
        capsize=3,
        label="last step, with 2 standard errors either side",
    )
    a.axvline(0, color="grey", lw=1)
    a.invert_yaxis()
    a.set_xscale("symlog", linthresh=1)
    a.set_xlabel("points of test accuracy (symlog)")
    a.set_title(ds)
    a.grid(alpha=0.3, axis="x")
axs[0].legend(frameon=False, fontsize=8)
plt.suptitle(
    "Does the last step of each axis still pay? A bar whose whiskers cross 0 is no result"
)
plt.tight_layout()
plt.show()

**Reading.** Which axis binds is the one whose last step still pays by the
rule:

| | DTD (47 textures, 64 rows per class) | EuroSAT (10 land uses, 480 rows per class) |
|---|---|---|
| parameters | width +2.4 up to `1 x 512`, then +0.16, no result; the fourth layer −1.02, marginal | not measured |
| data, last doubling | **+3.1**, 10.9 se: still paying | +0.4, 3.5 se at equal steps (2.9 at equal epochs): still paying, a little |
| compute, epoch 20 to 80 | +0.11, 1.4 se: flat (from epoch 10 it still gained 0.57, 4.5 se) | +0.09, 1.3 se: flat |
| binds | **data** | data, a little |
| apart: encoder, best against second | +14.1, 59.8 se: the choice mattered a lot | +1.75, 10.2 se: the choice mattered |

On DTD one measured axis binds: more labelled rows, still three points per doubling. Apart
from that, the choice of encoder mattered a lot (part of `dinov2-small`'s lead may come from
DINOv2's curation, 2.5). On EuroSAT what still pays is small: the last doubling of the data
buys under half a point, compute is flat, and the head was not measured there. Of the three
encoders the best is already in use; what could still pay is an encoder better than these
three, which this notebook does not measure. The same protocol on two image caches gives two
different answers, and that is the point: **the axis that pays differs per problem, and you
find out which by drawing the curves.**

**Capacity is not the resource. Information is.** Data adds information; a better encoder
adds information. Parameters and epochs only let you use information the vectors already
hold, and on a frozen representation that is used up early.

## 2.7 Reading a scaling-law figure: Kaplan's three laws

**Question.** 2.1 to 2.6 drew curves on one frozen encoder. The scaling-law papers draw the
same axes for models trained from scratch, over far wider ranges, and fit a law to them.
How do you read such a figure, and what does it not promise? The worked example is Kaplan
et al. (2020), *Scaling Laws for Neural Language Models* (the PDF in `docs/papers/`).

**What was measured.** Transformer language models trained on WebText2 (Section 2.3). The
score is the **test loss**: the cross-entropy per token on held-out text, in **nats** (the
natural logarithm, as `nn.CrossEntropyLoss` uses; Section 1.3); lower is better. Three resources, each with its unit (Section 1.3):

- **parameters $N$**, non-embedding: the weights without the embeddings, the lookup tables
  that turn each token into a vector;
- **dataset size $D$**, in tokens (a token is a word or a piece of one);
- **compute $C_{min}$**, in PF-days: $10^{15}$ floating point operations per second for a
  day, $8.64 \cdot 10^{19}$ operations. $C_{min}$ is Kaplan's estimate of the least compute
  that reaches a given loss: the compute of training with a small enough batch (Section 1.3).

Each law is a power law, $L(X) = (X_c / X)^{\alpha}$, and holds only when the other two
resources are not the bottleneck (Section 1.2, Figure 1 caption):

| law | holds for (Section 1.2) | $\alpha$ | $X_c$ | equation |
|---|---|---|---|---|
| $L(N)$ | models trained to convergence on enough data | 0.076 | $8.8 \cdot 10^{13}$ parameters | 1.1 |
| $L(D)$ | large models on a limited dataset, early stopping | 0.095 | $5.4 \cdot 10^{13}$ tokens | 1.2 |
| $L(C_{min})$ | enough data, a model of the best size, a small enough batch | 0.050 | $3.1 \cdot 10^{8}$ PF-days | 1.3 |

Take the log of both sides: $\log L = \alpha \log X_c - \alpha \log X$, a straight line in
$\log X$. So on log-log axes a power law is a **straight line with slope $-\alpha$**. The
slope is what to read: 10× more of a resource multiplies the loss by $10^{-\alpha}$. The constants $X_c$
depend on the tokenization and have no meaning of their own (Section 1.2).

Each line below is drawn only over the runs behind it in Figure 1: compute from about
$8 \cdot 10^{-9}$ to $0.25$ PF-days, dataset size from $2.1 \cdot 10^{7}$ to
$1.4 \cdot 10^{9}$ tokens (Figure 9 lists these sizes), parameters from about
$6 \cdot 10^{3}$ to $1.5 \cdot 10^{9}$. The compute and parameter bounds are read off the
figure's axes, so they are approximate. (Figure 1's legend prints $C_c = 2.3 \cdot 10^{8}$;
equation 1.3 and Table 5 give $3.1 \cdot 10^{8}$, used here. The difference is 1.5% of the
loss.)

In [ ]:
# Kaplan et al. (2020): alpha and X_c from eq. 1.1 to 1.3 (the same values in Table 5);
# lo and hi are the measured range of each law, read off Figure 1.
KAPLAN = {
    "C": {
        "alpha": 0.050,
        "Xc": 3.1e8,
        "lo": 8e-9,
        "hi": 0.25,
        "eq": "1.3",
        "xlabel": "compute $C_{min}$ (PF-days, non-embedding)",
    },
    "D": {
        "alpha": 0.095,
        "Xc": 5.4e13,
        "lo": 2.1e7,
        "hi": 1.4e9,
        "eq": "1.2",
        "xlabel": "dataset size $D$ (tokens)",
    },
    "N": {
        "alpha": 0.076,
        "Xc": 8.8e13,
        "lo": 6e3,
        "hi": 1.5e9,
        "eq": "1.1",
        "xlabel": "parameters $N$ (non-embedding)",
    },
}


def kaplan_loss(axis: str, x) -> np.ndarray:
    # Test loss in nats per token from the fitted power law L = (X_c / x)^alpha.
    k = KAPLAN[axis]
    return (k["Xc"] / np.asarray(x, dtype=float)) ** k["alpha"]


fig, axs = plt.subplots(1, 3, figsize=(13, 4), sharey=True)
for a, (axis, k) in zip(axs, KAPLAN.items()):
    x = np.geomspace(k["lo"], k["hi"], 200)
    a.plot(x, kaplan_loss(axis, x), color="tab:blue")
    a.plot(
        [k["lo"], k["hi"]],
        kaplan_loss(axis, [k["lo"], k["hi"]]),
        "o",
        color="tab:blue",
        label="ends of the measured range",
    )
    a.set_xscale("log")
    a.set_yscale("log")
    a.set_xlabel(k["xlabel"])
    mant, exp = f"{k['Xc']:.1e}".split("e")
    sym = "C_{min}" if axis == "C" else axis
    a.set_title(
        f"$L = ({mant} \\cdot 10^{{{int(exp)}}} / {sym})^{{{k['alpha']}}}$, eq. {k['eq']}",
        fontsize=10,
    )
    a.grid(alpha=0.3, which="both")
    # every panel spans the same 8 decades, so the angles of the lines compare by eye
    mid = (np.log10(k["lo"]) + np.log10(k["hi"])) / 2
    a.set_xlim(10 ** (mid - 4), 10 ** (mid + 4))
ticks = [2.4, 2.6, 2.8, 3.0, 3.5, 4.0, 4.5, 5.0, 6.0, 7.0]
axs[0].set_yticks(ticks)
axs[0].set_yticklabels([f"{t:g}" for t in ticks])
axs[0].yaxis.set_minor_formatter(plt.NullFormatter())
axs[0].set_ylabel("test loss (nats per token, log)")
axs[0].legend(frameon=False, fontsize=8)
plt.suptitle(
    "Kaplan et al. (2020): one straight line per resource, drawn only where measured; every panel spans 8 decades"
)
plt.tight_layout()
plt.show()

Three straight lines: each resource, grown by a factor, cuts the loss by a fixed factor,
over the whole measured range. Every panel spans the same eight decades on its x axis, so
the angles of the lines can be compared by eye; on panels with different ranges they could
not.

**Expect.** Read two values off the figure before you run the next cell: the test loss of
a model with $N = 10^{8}$ parameters (right panel), and of a large model trained on
$D = 10^{9}$ tokens (middle panel). Then: which line is steepest, so buys the most per 10×?

In [ ]:
for axis, x in [("N", 1e8), ("D", 1e9)]:
    print(f"L({axis} = {x:.0e}) = {float(kaplan_loss(axis, x)):.2f} nats per token")
for axis, k in KAPLAN.items():
    print(f"10x more {axis}: loss x 10^-{k['alpha']} = {10 ** -k['alpha']:.2f}")

10× the data multiplies the loss by 0.80, 10× the parameters by 0.84, 10× the compute by
0.89: the steeper the line, the more 10× buys.

**What the figure does not promise.** Outside its measured range a line is a formula, not
a measurement. The next cell asks the parameter law about models far larger than any that
was trained.

In [ ]:
for n in [1.5e9, 1e12, 1e15, 1e20]:
    where = "inside" if n <= KAPLAN["N"]["hi"] else "outside"
    print(
        f"N = {n:.1e}: L = {float(kaplan_loss('N', n)):.2f} nats per token ({where} the measured range)"
    )

The formula keeps falling, below 1 nat per token and towards 0. Kaplan et al. say the loss
must flatten out before it reaches zero (Section 1.1), because text cannot be predicted
perfectly (Section 6.3); where the bend comes, the figure does not say.

**When another resource is the bottleneck,** the line does not apply either. Equation 1.5
fits $N$ and $D$ together, with its own constants (Table 2):
$L(N, D) = \left[(N_c/N)^{\alpha_N/\alpha_D} + D_c/D\right]^{\alpha_D}$. The next cell
compares growing models on the full 22 billion tokens of WebText2 and on 86 million (two
of the dataset sizes of Figure 9).

In [ ]:
# eq. 1.5 with the constants of its own fit, Table 2 of Kaplan et al. (2020)
ND = {"aN": 0.076, "aD": 0.103, "Nc": 6.4e13, "Dc": 1.8e13}


def kaplan_loss_nd(n, d) -> float:
    # Early-stopped test loss of a model with n parameters on d tokens, eq. 1.5.
    return float(((ND["Nc"] / n) ** (ND["aN"] / ND["aD"]) + ND["Dc"] / d) ** ND["aD"])


print(f"{'N':>8s} {'L(N), eq. 1.1':>14s} {'D = 22e9':>9s} {'D = 86e6':>9s}")
for n in [1e6, 1e7, 1e8, 1e9]:
    print(
        f"{n:8.0e} {float(kaplan_loss('N', n)):14.2f} "
        f"{kaplan_loss_nd(n, 22e9):9.2f} {kaplan_loss_nd(n, 86e6):9.2f}"
    )

On 22 billion tokens the loss falls with $N$ about as the parameter law says; it lies a
little below the law's column because equation 1.5 is a separate fit with its own
constants, not the same line. On 86 million tokens the loss hardly moves past $10^{7}$
parameters: the data is the bottleneck.

## 2.8 The same law in other domains, with other slopes

**Question.** Is a power law something of language models, or of one paper?

Two other studies measured it in other domains. Hestness et al. (2017), *Deep Learning
Scaling is Predictable, Empirically* (arXiv:1712.00409), grew the training set and fitted
the validation error as $\varepsilon(m) \propto m^{\beta_g}$, with $m$ the number of
training examples. Henighan et al. (2020), *Scaling Laws for Autoregressive Generative
Modeling* (arXiv:2010.14701), grew the model and fitted
$L(N) = L_\infty + (N_0/N)^{\alpha_N}$: a power law on top of an irreducible loss
$L_\infty$ that no model gets below. Neither PDF is in `docs/papers/`. The next cell lists
their exponents; a negative exponent of $-0.3$ means 10× more multiplies the error by
$10^{-0.3} = 0.50$.

In [ ]:
# study, domain, what grows, what falls, exponent, where in the paper
SLOPES = [
    (
        "Hestness 2017",
        "translation (WMT'16 De-En)",
        "data",
        "per-token error",
        -0.128,
        "Sec. 4.1, projected",
    ),
    (
        "Hestness 2017",
        "word language model",
        "data",
        "cross-entropy",
        -0.066,
        "Table 1",
    ),
    (
        "Hestness 2017",
        "character language model",
        "data",
        "cross-entropy",
        -0.094,
        "Table 1",
    ),
    (
        "Hestness 2017",
        "ImageNet classification",
        "data",
        "top-1 error",
        -0.309,
        "Sec. 4.3",
    ),
    (
        "Hestness 2017",
        "speech (Deep Speech 2)",
        "data",
        "character error rate",
        -0.299,
        "Sec. 4.4",
    ),
    ("Kaplan 2020", "language model (WebText2)", "data", "loss", -0.095, "eq. 1.2"),
    (
        "Kaplan 2020",
        "language model (WebText2)",
        "parameters",
        "loss",
        -0.076,
        "eq. 1.1",
    ),
    ("Henighan 2020", "images 8x8", "parameters", "reducible loss", -0.24, "Table 1"),
    ("Henighan 2020", "images 32x32", "parameters", "reducible loss", -0.13, "Table 1"),
    (
        "Henighan 2020",
        "video (VQ 16x16x16)",
        "parameters",
        "reducible loss",
        -0.24,
        "Table 1",
    ),
    (
        "Henighan 2020",
        "text-to-image (image part)",
        "parameters",
        "reducible loss",
        -0.16,
        "Table 1",
    ),
    (
        "Henighan 2020",
        "math problems (extrapolate)",
        "parameters",
        "reducible loss",
        -0.16,
        "Table 1",
    ),
]

print(
    f"{'study':14s} {'domain':28s} {'grows':10s} {'falls':20s} {'exponent':>8s} {'x per 10x':>9s}  source"
)
for study, domain, grows, falls, b, src in SLOPES:
    print(
        f"{study:14s} {domain:28s} {grows:10s} {falls:20s} {b:8.3f} {10**b:9.2f}  {src}"
    )

**Reading.** In every domain the error or loss falls as a power law of the resource, but
the slope differs: 10× the data multiplies the error of a word language model by 0.86 and
the top-1 error of ImageNet classification (the share of images whose highest-scoring
class is wrong) by 0.49. Hestness et al. say it themselves:
different applications yield different exponents, and their data exponents mostly lie
between −0.07 and −0.35 (Section 1). Henighan et al. find a different exponent for each
image resolution (Section 1.1). The translation exponent is the projection of Hestness's
best-fit curve (Section 4.1); single models gave −0.36 and −0.30. Within one domain, Hestness et al. found that better
models shift the curve but leave the exponent as it was (abstract).

Read the table with care: the metrics differ (an error rate, a cross-entropy, a reducible
loss), and so do the models and the training. The exponents say that the law has the same
form everywhere; they do not carry a slope from one domain to another. **The slope of your
problem is measured on your problem.**

## 2.9 Your own slope: DTD against EuroSAT

**Question.** The data axes of 2.3 and 2.6 are curves of your own. Are they straight lines
on log-log axes, and with what slope?

**Setup.** A scaling law is about something that falls towards zero. The laws are written
for a loss; here the **test error**, 1 − accuracy, stands in for it. The next cell draws
that error of the equal-steps arm (only the data changes)
against the number of training rows, both on log axes, and fits a straight line per
dataset: $\log_{10}(\text{error}) = c + \beta \log_{10}(\text{rows})$. It prints the slope
$\beta$ with its standard error (from how far the points lie off the line), what 10× more
rows multiplies the error by, how far each point lies off the line, and the slope of the
last doubling alone with its standard error (from the seeds). Then it reads the
differences with the rule. The parameter axis of 2.2 is not fitted: with
depth it goes down again, and no straight line describes that.

**Expect.** DTD's accuracy moved 20.5 points over its data axis, EuroSAT's 5.7. Write down
before you run it: will the two slopes on log-log axes match, or is DTD's much steeper?

In [ ]:
def own_slope(rows):
    """Rows, test error, its std, the fitted slope beta, its standard error, and c."""
    n = np.array([r[0] for r in rows], dtype=float)
    err = 1 - np.array([r[3] for r in rows])
    sd_err = np.array([r[4] for r in rows])
    x, y = np.log10(n), np.log10(err)
    beta, c = np.polyfit(x, y, 1)
    if len(n) > 2:
        resid = y - (c + beta * x)
        se = float(np.sqrt(resid @ resid / (len(n) - 2) / ((x - x.mean()) ** 2).sum()))
    else:
        se = float("nan")
    return n, err, sd_err, float(beta), se, float(c)


def last_doubling(rows) -> tuple[float, float]:
    """Slope of log10(error) over the last doubling, and its standard error from the seeds."""
    d, se, _ = compare(np.log10(1 - rows[-2][10]), np.log10(1 - rows[-1][10]))
    step = np.log10(rows[-1][0] / rows[-2][0])
    return d / step, se / step


fits = {}
plt.figure(figsize=(7, 4))
for (name, rows), color in zip(
    [("DTD", dtd_data), ("EuroSAT", eurosat_data)], ["C0", "C1"]
):
    n, err, sd_err, beta, se, c = own_slope(rows)
    off = np.log10(err) - (c + beta * np.log10(n))
    last, se_last = last_doubling(rows)
    fits[name] = (beta, se)
    z_bend = (last - beta) / np.sqrt(se_last**2 + se**2)
    print(
        f"{name}: {int(n[0])} to {int(n[-1])} rows ({np.log10(n[-1] / n[0]):.1f} decades)"
    )
    print(
        f"  slope {beta:.3f} (se {se:.3f}): 10x the rows multiplies the error by {10**beta:.2f}"
    )
    print(f"  off the line (log10 error): {', '.join(f'{o:+.3f}' for o in off)}")
    print(
        f"  slope of the last doubling alone: {last:.3f} (se {se_last:.3f}); "
        f"against the fit {z_bend:+.1f} se, {verdict(z_bend)}"
    )
    plt.errorbar(
        n, err, yerr=sd_err, fmt="o", capsize=3, color=color, label=f"{name}, measured"
    )
    plt.plot(
        n,
        10 ** (c + beta * np.log10(n)),
        color=color,
        alpha=0.6,
        label=f"{name}, fit: slope {beta:.2f}",
    )
(b1, s1), (b2, s2) = fits["DTD"], fits["EuroSAT"]
z = (b2 - b1) / np.sqrt(s1**2 + s2**2)
print(
    f"EuroSAT's slope minus DTD's: {b2 - b1:+.3f}, se {np.sqrt(s1**2 + s2**2):.3f}, "
    f"{z:+.1f} se, {verdict(z)}"
)
plt.xscale("log")
plt.yscale("log")
plt.xlabel("training rows (log)")
plt.ylabel("test error, 1 - accuracy (log)")
ax = plt.gca()
ax.set_xticks([150, 300, 1000, 3000])
ax.set_xticklabels(["150", "300", "1000", "3000"])
ax.set_yticks([0.05, 0.07, 0.1, 0.15, 0.2, 0.3, 0.4])
ax.set_yticklabels(["0.05", "0.07", "0.10", "0.15", "0.20", "0.30", "0.40"])
ax.xaxis.set_minor_formatter(plt.NullFormatter())
ax.yaxis.set_minor_formatter(plt.NullFormatter())
plt.title("The data axis on log-log axes: one slope per dataset")
plt.grid(alpha=0.3, which="both")
plt.legend(frameon=False, fontsize=8)
plt.tight_layout()
plt.show()

**Reading.** DTD's slope is −0.232 (se 0.008), EuroSAT's −0.262 (se 0.020).
Their difference, −0.029, is 1.4 standard errors: no result. On log-log axes the two slopes
cannot be told apart, although DTD's accuracy moved almost four times as many points. 10×
the rows multiplies DTD's error by 0.59 and EuroSAT's by 0.55; the same factor takes fewer
points off EuroSAT's error because that error is four times smaller to begin with.

With five points per line the fit keeps 3 degrees of freedom, so its standard errors are
rough, and the thresholds of 2 and 3 with them. The bend test compares the last doubling
with a fit that contains those same two points, so the two are not independent.

The lines fit differently. DTD's points lie within 0.01 of its line (in $\log_{10}$ of the
error), and its last doubling alone has slope −0.21, no result against the fit (1.2 se).
EuroSAT's points lie up to 0.022 off, and its last doubling has slope −0.12, a result
against the fit (3.7 se): its curve bends at the end, and a straight line no longer
describes it there.

**How this fits with "the slope differs per domain".** DTD and EuroSAT are one kind of task:
image classification on one frozen encoder, over 1.3 decades of rows. Two slopes that cannot
be told apart here do not show that the domains of 2.8 share a slope; they show that these
two problems, measured this way, do not separate. And the 20.5 against 5.7 points of 2.6 are
points of accuracy, not the factor by which the error falls: the same factor takes more
points off a larger error.

**A local description, not a law.** These slopes cover 1.3 decades of rows; Kaplan's data
law covers 1.8 decades of tokens, and Hestness et al. grew ImageNet from 1 to 800 images
per class (Section 4.3). And this is a frozen encoder with a small head, not a model
trained from scratch: the encoder already learned from far more images, and only the head
sees these rows. The slope says how the error moved over the rows you measured, on these
vectors. It does not say what 10× more rows will bring; that is a measurement too.

## 2.10 A bigger model needs more data: the overfitting ratio

**Question.** Kaplan et al. measured that a bigger model trained on the same data overfits
more. (2.2 showed depth costing points on 3008 rows, but did not test whether that was
overfitting.) How much more data keeps the overfitting where it was?

Kaplan et al. define the **overfitting penalty** as how far the loss on $D$ tokens sits
above the loss with unlimited data: $\delta L = L(N, D) / L(N, \infty) - 1$ (equation 4.2).
Equation 1.5 gives $L(N, \infty) = (N_c/N)^{\alpha_N} = \left[(N_c/N)^{\alpha_N/\alpha_D}\right]^{\alpha_D}$,
so in one step

$$\frac{L(N, D)}{L(N, \infty)} = \frac{\left[(N_c/N)^{\alpha_N/\alpha_D} + D_c/D\right]^{\alpha_D}}{\left[(N_c/N)^{\alpha_N/\alpha_D}\right]^{\alpha_D}} = \left[1 + \left(\frac{N}{N_c}\right)^{\alpha_N/\alpha_D} \frac{D_c}{D}\right]^{\alpha_D},$$

and $\delta L$ is that minus 1 (equation 4.3). It depends on $N$ and $D$ only through
$N^{\alpha_N/\alpha_D} / D$. Keep that ratio fixed and the penalty stays the same:
the data has to grow as $D \propto N^{0.076/0.103} = N^{0.74}$ (Section 1.2).

**Expect.** The model grows 8×. Write down: by what factor must the data grow to keep the
penalty equal, and what happens to the penalty if the data stays the same?

In [ ]:
exponent = ND["aN"] / ND["aD"]


def penalty(n, d) -> float:
    # Overfitting penalty delta L of eq. 4.3: the loss on d tokens over the loss with unlimited data, minus 1.
    return float((1 + (n / ND["Nc"]) ** exponent * ND["Dc"] / d) ** ND["aD"] - 1)


grow = 8**exponent
print(f"alpha_N / alpha_D = {ND['aN']} / {ND['aD']} = {exponent:.2f}")
print(f"8x the parameters asks for 8^{exponent:.2f} = {grow:.1f}x the data\n")

n0, d0 = 1e8, 1e9
for label, n, d in [
    ("start", n0, d0),
    ("8x N, same D", 8 * n0, d0),
    (f"8x N, {grow:.1f}x D", 8 * n0, grow * d0),
]:
    print(
        f"{label:16s} N = {n:.1e}  D = {d:.1e}   penalty {100 * penalty(n, d):.2f}% of the loss"
    )

**Reading.** With the same data, the 8× bigger model's penalty rises from 7.0% to 18.8% of
the loss: it overfits. With $8^{0.74} = 4.6$ times the data it is back at 7.0%. **8× the
weights asks about 5× the data**, the number Kaplan et al. give in Section 1.1.

**The ratio belongs to the study.** Kaplan's ratio comes from their language models,
regularised with 10% dropout that was not tuned while data and model size varied (Section
4.2). Hestness et al. measured the relation the other way round: the model size that fits
a training set best grows as $m^{\beta_p}$, with $\beta_p = 0.69$ for word language models
(Section 4.2.1; 0.78 for their deeper model) and $0.573$ for ImageNet (Section 4.3). The
next cell turns those around, and adds the ratio of the compute question.

In [ ]:
print("equal overfitting penalty (Kaplan, eq. 4.3):")
print(f"  D ~ N^{exponent:.2f}: 8x the parameters asks for {8**exponent:.1f}x the data")
print("best model size for a training set (Hestness, N ~ m^beta_p):")
for study, beta_p in [("word language model", 0.69), ("ImageNet", 0.573)]:
    print(
        f"  {study:20s} beta_p = {beta_p}: 8x the parameters goes with {8 ** (1 / beta_p):.0f}x the data"
    )
print("the best split of a compute budget (N ~ C^a, D ~ C^b, so D ~ N^(b/a)):")
for study, a, b in [
    ("Kaplan", 0.73, 0.27),
    ("Hoffmann, approach 1", 0.50, 0.50),
    ("Hoffmann, approach 2", 0.49, 0.51),
    ("Hoffmann, approach 3", 0.46, 0.54),
]:
    print(
        f"  {study:20s} a = {a}, b = {b}: 8x the parameters goes with {8 ** (b / a):.1f}x the data"
    )

Each ratio answers its own question, and only ratios that answer the same question can be
compared:

- **equal overfitting penalty**: Kaplan's 4.6×, the least data that keeps a bigger model
  from overfitting more;
- **the best split of a compute budget**: here two studies answer the same question
  differently. Kaplan et al. found $N \propto C^{0.73}$, $D \propto C^{0.27}$ (equation 6.1,
  Table 6), so 8× the parameters goes with 2.2× the data. Hoffmann et al. (2022,
  arXiv:2203.15556), who matched the learning-rate schedule (how the step size is lowered
  during training) to each run's number of training tokens, found $a$ and $b$ near 0.5
  (their Table 2): 8× the parameters goes with 8× to 11.5× the data;
- **the best model size for a training set**: Hestness's 20× and 38×, a third question.

So "how much more data?" has no answer of its own: it depends on the question, the models
and how they were trained. Read a ratio as a property of the study it came from.

## What you now know

- A **scaling curve** is the score measured against one resource with the others held
  fixed, every point trained over several seeds. A difference counts by lesson 2's rule:
  within 2 standard errors no result, 2 to 3 marginal. The seeds do not vary the split, so
  the spread is a lower bound.
- Choices (the epoch count, each head's and each encoder's epoch, the head to prefer) are
  made on **valid**; the curves and every number quoted are scored on **test**, which no
  choice saw.
- On DTD with a frozen `dinov2-small`: width buys about 2.5 points up to `1 x 512` and
  then stops, depth gives them back, compute pays up to about epoch 20 of one head and is
  flat after it, data still pays about 3 points per doubling at the full split, and the
  encoder spans 15 points. On EuroSAT the same axes move far less: the last doubling of the
  data still pays, but under half a point. The axis that pays differs per problem.
- A conclusion can go beyond the top score: chosen on valid, `1 x 256` was no result against the best head
  with a quarter of its parameters, and on this machine answered about three times as fast;
  on test it cost 0.6 points, a price you state with the choice. A claim about latency is timed on the
  hardware it is about; a claim about energy needs an energy measurement.
- Scaling laws say the test loss falls as a power of compute (PF-days), dataset size
  (tokens) and non-embedding parameters, as long as the other two are not the bottleneck.
  On log-log axes that is a straight line; its slope says what 10× more buys, and only
  inside the measured range. The law has been measured in language, images, speech, video
  and more, with a different slope per domain. On DTD and EuroSAT your own slopes were −0.23 and −0.26, no result
  against each other, over 1.3 decades of rows on a frozen encoder, and EuroSAT's curve bends
  at its end: a local description, not a law.
- Kaplan measured that a bigger model on the same data overfits more (in 2.2 depth cost
  points on DTD; whether that was overfitting was not tested). From a measured ratio you
  read how much more data a bigger model needs: in Kaplan's study 8× the weights asks about
  5× the data to keep the overfitting penalty equal. The ratio differs per question and per
  study.

That is the question to carry into notebook 03: if the axes that matter are not the ones
you were going to search over, **what should a search search?**

## Exercises

**E1 — where does data stop paying?** 2.3 says on DTD more rows still pay at the full
split, and 2.6 says on EuroSAT they pay far less. Fit a line to accuracy against log(rows)
over the first three and over the last three points of each equal-steps curve, and compare
the slopes. At what slope would you stop labelling, and who decides that number?

**E2 — does the parameter axis change with less data?** Repeat 2.2 at 10% of the train
rows (300 rows, `rows=300` in `repeat`). Predict first: does capacity matter more when data
is scarce, or less? Then check the error bars before believing the ranking.

**E3 — the encoder axis upwards.** DTD also has caches of `dinov2-base` and
`dinov2-large` (`pttrn-io/dtd-dinov2-base`, `pttrn-io/dtd-dinov2-large`). Add them to 2.5.
Does the encoder axis still pay above `dinov2-small`, by lesson 2's rule? Does that
change which axis binds on DTD?

**E4 — a curve that bends.** 2.9 fits a straight line on log-log axes. Fit
`error = gamma + alpha * rows^beta` instead (an offset `gamma` for the error no amount of
data removes, as Hestness et al. do) to the equal-steps curves of DTD and EuroSAT. Does the
offset explain EuroSAT's flattening? Extrapolate both fits to 10× the rows and argue,
from the residuals, which extrapolation you believe.

**E5 — the budget of the data axis.** 2.3 gives every subset either `EPOCHS` epochs or
the steps of the full set. Give each subset its own epoch count instead, chosen on valid by
the rule of 2.1 (as 2.2 does per head and 2.5 per encoder), and draw that third curve next
to the other two. Which of the three would you report as "the data axis", and why?

**E6 — read the ratios.** With the cells of 2.10: how much more data keeps the
overfitting penalty equal when the model grows 64×, by Kaplan's ratio and by Hestness's
two? Which would you use for the heads of 2.2, and why none of them may apply?

**E7 — latency beyond one machine.** 2.2 timed two heads over the whole test set as one
batch. Time them one row at a time as well, and add `4 x 256`. Does the order by parameter
count match the order by latency, by more than the spread of the repeats? What would you
have to measure before claiming the same on a phone, and what before claiming anything
about energy?